In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "study_hour": [1, 2, 3, 4, 5, 6, 7, 8, 2, 4, 6, 8],
    "sleep_hour": [5, 6, 5, 7, 6, 8, 7, 8, 4, 5, 6, 7],
    "device": [
        "mobile", "pc", "mobile", "pc",
        "tablet", "pc", "mobile", "pc",
        "tablet", "mobile", "pc", "tablet"
    ],
    "passed": [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
})

In [4]:
df.head()

,study_hour,sleep_hour,device,passed
0,1,5,mobile,0
1,2,6,pc,0
2,3,5,mobile,0
3,4,7,pc,0
4,5,6,tablet,1


In [8]:
print("shape\n",df.shape)
print()
print("dtypes\n",df.dtypes)

shape
 (12, 4)

dtypes
 study_hour    int64
sleep_hour    int64
device          str
passed        int64
dtype: object


In [10]:
df["study_hour"].value_counts()

study_hour
2    2
4    2
6    2
8    2
1    1
3    1
5    1
7    1
Name: count, dtype: int64

In [14]:
df["passed"].value_counts()

passed
0    6
1    6
Name: count, dtype: int64

In [11]:
df["device"].value_counts()

device
pc        5
mobile    4
tablet    3
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df[df.drop(columns=["passed"]).columns], df["passed"],
    random_state=42,
    stratify=df["passed"]
)

In [30]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = ["study_hour", "sleep_hour"]
ord_cols = ["device"]

pipeline = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("ord", OneHotEncoder(handle_unknown="error"), ord_cols)
])

X_train_scaled = pipeline.fit_transform(X_train)
X_test_scaled = pipeline.transform(X_test)
X_train_scaled.dtype

dtype('float64')

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn import set_config

set_config(display="text")

model = Pipeline([
    ("preprocess", pipeline),
    ("model", LogisticRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['study_hour', 'sleep_hour']),
                                                 ('ord', OneHotEncoder(),
                                                  ['device'])])),
                ('model', LogisticRegression())])

In [34]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pred = model.predict(X_test)

print(classification_report(y_test, pred))
print()
print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.50      1.00      0.67         1

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3


0.6666666666666666
[[1 1]
 [0 1]]


In [36]:
print(model)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['study_hour', 'sleep_hour']),
                                                 ('ord', OneHotEncoder(),
                                                  ['device'])])),
                ('model', LogisticRegression())])
